<a href="https://colab.research.google.com/github/NikhilKumarReddyKadiveti/MLA0204/blob/main/Practical%20Problem%202.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Candidate-Elimination Algorithm

Now, let's implement and demonstrate the Candidate-Elimination algorithm using the `enjoysport.csv` data. This algorithm finds all hypotheses consistent with the training examples by maintaining a general (G) boundary and a specific (S) boundary.

In [11]:
import pandas as pd # Ensure pandas is imported for general use here

def candidate_elimination(training_data):
    num_attributes = len(training_data[0]) - 1
    # Initialize G with the most general hypothesis
    G = [['?' for _ in range(num_attributes)]]
    # Initialize S with the most specific hypothesis
    S = [['0' for _ in range(num_attributes)]]

    print("\nInitial G:", G)
    print("Initial S:", S)

    # Collect attribute domains from the training data for generate_minimal_specializations
    domains = [set() for _ in range(num_attributes)]
    for row in training_data:
        for attr_idx in range(num_attributes):
            domains[attr_idx].add(row[attr_idx])

    for i, example in enumerate(training_data):
        attributes = example[:-1]  # All but the last element are attributes
        target = example[-1]      # The last element is the target classification

        print(f"\nProcessing Example {i+1}: {example}")

        if target == 'yes': # Positive example
            # Remove any hypothesis in G that is inconsistent with the positive example
            G = [g for g in G if is_consistent(g, attributes)]

            # For each hypothesis s in S that is inconsistent with the positive example
            temp_S = []
            for s in S:
                if not is_consistent(s, attributes): # 's' is too specific, needs generalization
                    generalizations = generate_minimal_generalizations(s, attributes)
                    for h in generalizations:
                        # 'h' must be consistent with the positive example (already handled by generate_minimal_generalizations)
                        # And 'h' must be more specific than or equal to ALL 'g' in G
                        if all(is_more_specific(h, g) or h == g for g in G): # This is the crucial filter against G
                            temp_S.append(h)
                else: # 's' is consistent with the positive example
                    temp_S.append(s)
            S = remove_more_general_hypotheses(temp_S) # Remove duplicates and more general hypotheses

        else: # Negative example ('no')
            # Remove any hypothesis in S that is inconsistent with the negative example
            S = [s for s in S if not is_consistent(s, attributes)]

            # For each hypothesis g in G inconsistent with the negative example
            temp_G = []
            for g in G:
                if is_consistent(g, attributes): # 'g' is too general, covers negative example, needs specialization
                    specializations = generate_minimal_specializations(g, attributes, domains)
                    for h in specializations:
                        # 'h' must be inconsistent with the negative example (handled by generate_minimal_specializations)
                        # And 'h' must NOT be more specific than any 's' in S (i.e., not eliminated by S).
                        # This ensures 'h' stays above or at the S-boundary.
                        if not any(is_more_specific(h, s) and h != s for s in S): # Crucial filter against S
                            temp_G.append(h)
                else: # 'g' is consistent (does NOT cover negative example)
                    temp_G.append(g)
            G = remove_more_specific_hypotheses(temp_G) # Remove duplicates and more specific hypotheses

        print(f"   S{i+1}:", S)
        print(f"   G{i+1}:", G)

    return S, G

def is_consistent(hypothesis, example):
    # A hypothesis is consistent with an example if it 'covers' the example
    for i in range(len(hypothesis)):
        if hypothesis[i] != '?' and hypothesis[i] != example[i]:
            return False
    return True

def is_more_specific(h1, h2):
    # h1 is more specific than h2 if h1 covers a subset of what h2 covers
    # Returns True if h1 is more specific or equally specific as h2
    for i in range(len(h1)):
        if h1[i] == '?' and h2[i] != '?': # h1 is more general than h2 at this position
            return False
        if h1[i] != h2[i] and h1[i] != '?' and h2[i] != '?': # different specific values
            return False
    return True

def is_more_general(h1, h2):
    # h1 is more general than h2 if h1 covers a superset of what h2 covers
    # Returns True if h1 is more general or equally general as h2
    return is_more_specific(h2, h1)

def generate_minimal_generalizations(s_hypothesis, positive_example):
    generalizations = []
    for i in range(len(s_hypothesis)):
        if s_hypothesis[i] == '0': # If specific (0), generalize to example's value
            new_s = list(s_hypothesis)
            new_s[i] = positive_example[i]
            generalizations.append(new_s)
        elif s_hypothesis[i] != '?' and s_hypothesis[i] != positive_example[i]:
            new_s = list(s_hypothesis)
            new_s[i] = '?' # Generalize to '?' if attributes differ
            generalizations.append(new_s)
    # Filter to ensure only unique minimal generalizations
    unique_generalizations = []
    for gen in generalizations:
        if gen not in unique_generalizations:
            unique_generalizations.append(gen)
    return unique_generalizations

def generate_minimal_specializations(g_hypothesis, negative_example, domains):
    specializations = []
    for i in range(len(g_hypothesis)):
        # Case 1: Attribute is '?' (most general), so it must be specialized
        if g_hypothesis[i] == '?':
            for val in domains[i]:
                # Create specialization by replacing '?' with specific values from its domain
                # that do NOT match the negative example's value for this attribute.
                if val != negative_example[i]:
                    new_g = list(g_hypothesis)
                    new_g[i] = val
                    specializations.append(new_g)
        # Case 2: Attribute is specific and matches the negative example, so it must be changed
        elif g_hypothesis[i] == negative_example[i]:
            for val in domains[i]:
                if val != negative_example[i]: # Change to any other value in the domain
                    new_g = list(g_hypothesis)
                    new_g[i] = val
                    specializations.append(new_g)

    # Remove duplicates
    unique_specializations = []
    for spec in specializations:
        if spec not in unique_specializations:
            unique_specializations.append(spec)
    return unique_specializations

def remove_more_general_hypotheses(hypotheses):
    # For S, remove any hypothesis that is more general than another hypothesis in the set (i.e., keep most specific)
    non_redundant = []
    for h1 in hypotheses:
        is_redundant = False
        for h2 in hypotheses:
            if h1 != h2 and is_more_general(h1, h2): # If h1 is more general than h2, h1 is redundant
                is_redundant = True
                break
        if not is_redundant and h1 not in non_redundant: # Add only unique non-redundant
            non_redundant.append(h1)
    return non_redundant

def remove_more_specific_hypotheses(hypotheses):
    # For G, remove any hypothesis that is more specific than another hypothesis in the set (i.e., keep most general)
    non_redundant = []
    for h1 in hypotheses:
        is_redundant = False
        for h2 in hypotheses:
            if h1 != h2 and is_more_specific(h1, h2): # If h1 is more specific than h2, h1 is redundant
                is_redundant = True
                break
        if not is_redundant and h1 not in non_redundant: # Add only unique non-redundant
            non_redundant.append(h1)
    return non_redundant

# --- Rerun Data Preparation (for clarity and independence) ---
# The enjoysport.csv file from /content/enjoysport.csv is selected.
# Load the enjoysport.csv file
df_enjoysport = pd.read_csv('/content/enjoysport.csv')

# Convert all string data to lowercase for consistent comparison
df_enjoysport = df_enjoysport.apply(lambda x: x.astype(str).str.lower() if x.dtype == "object" else x)

print("Data from enjoysport.csv (lowercase for consistency):")
display(df_enjoysport.head())

# Convert DataFrame to the list of lists format expected by candidate_elimination
# Assuming the last column is the target ('enjoysport')
training_data_csv = df_enjoysport.values.tolist()

print("\nTraining data (list format) from enjoysport.csv:")
for row in training_data_csv:
    print(row)

# Run the Candidate-Elimination algorithm
final_S, final_G = candidate_elimination(training_data_csv)

print("\n--- Final Result for Candidate-Elimination (enjoysport.csv) ---")
print("\nThe Final Specific Boundary (S) is the set of all maximally specific hypotheses consistent with the positive examples:")
for s_hypothesis in final_S:
    print(f"  {s_hypothesis}")

print("\nThe Final General Boundary (G) is the set of all maximally general hypotheses consistent with the negative examples:")
for g_hypothesis in final_G:
    print(f"  {g_hypothesis}")

if not final_S and not final_G:
    print("\nNo consistent hypotheses found within the given examples.")
elif not final_S:
    print("\nNo specific hypotheses found consistent with the positive examples.")
elif not final_G:
    print("\nNo general hypotheses found consistent with the negative examples.")

# A simple way to visualize the version space (if it's not empty)
if final_S and final_G:
    print("\nThis represents the version space, containing all hypotheses consistent with the training data.")


Data from enjoysport.csv (lowercase for consistency):


,sky,airtemp,humidity,wind,water,forcast,enjoysport
0,sunny,warm,normal,strong,warm,same,yes
1,sunny,warm,high,strong,warm,same,yes
2,rainy,cold,high,strong,warm,change,no
3,sunny,warm,high,strong,cool,change,yes



Training data (list format) from enjoysport.csv:
['sunny', 'warm', 'normal', 'strong', 'warm', 'same', 'yes']
['sunny', 'warm', 'high', 'strong', 'warm', 'same', 'yes']
['rainy', 'cold', 'high', 'strong', 'warm', 'change', 'no']
['sunny', 'warm', 'high', 'strong', 'cool', 'change', 'yes']

Initial G: [['?', '?', '?', '?', '?', '?']]
Initial S: [['0', '0', '0', '0', '0', '0']]

Processing Example 1: ['sunny', 'warm', 'normal', 'strong', 'warm', 'same', 'yes']
   S1: [['sunny', '0', '0', '0', '0', '0'], ['0', 'warm', '0', '0', '0', '0'], ['0', '0', 'normal', '0', '0', '0'], ['0', '0', '0', 'strong', '0', '0'], ['0', '0', '0', '0', 'warm', '0'], ['0', '0', '0', '0', '0', 'same']]
   G1: [['?', '?', '?', '?', '?', '?']]

Processing Example 2: ['sunny', 'warm', 'high', 'strong', 'warm', 'same', 'yes']
   S2: [['sunny', 'warm', '0', '0', '0', '0'], ['sunny', '0', 'high', '0', '0', '0'], ['sunny', '0', '0', 'strong', '0', '0'], ['sunny', '0', '0', '0', 'warm', '0'], ['sunny', '0', '0', '0', 